In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
from math import ceil

In [ ]:
# Đường dẫn gốc đến thư mục chứa ảnh
base_dir = '/content/drive/MyDrive/Public'

# Đọc file CSV
csv_file = '/content/drive/MyDrive/CS114.P11.CarData/CarDataset-Splits-1-Train.csv'
car_dataset = pd.read_csv(csv_file)

# Kết hợp đường dẫn ảnh
car_dataset["FullImagePath"] = car_dataset["ImageFullPath"].apply(lambda x: os.path.join(base_dir, x))

# Khởi tạo mô hình MobileNet
model = MobileNet(weights="imagenet", include_top=False, pooling="avg")

<ipython-input-3-98642ed067ed>:12: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNet(weights="imagenet", include_top=False, pooling="avg")


17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Hàm xử lý ảnh và rút trích đặc trưng
def extract_features(image_path, model, target_size=(224, 224)):
    try:
        img = load_img(image_path, target_size=target_size)  # Tải ảnh và resize
        img_array = img_to_array(img)  # Chuyển ảnh thành numpy array
        img_array = np.expand_dims(img_array, axis=0)  # Thêm chiều batch
        img_array = preprocess_input(img_array)  # Tiền xử lý cho MobileNet
        features = model.predict(img_array)  # Rút trích đặc trưng
        return features.flatten()  # Trả về vector 1 chiều
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

In [ ]:
# Lưu kết quả toàn cục vào một danh sách
all_results = []

# Lọc ra tất cả các đường dẫn ảnh từ toàn bộ dataset
image_paths = car_dataset["FullImagePath"].tolist()

# Rút trích đặc trưng từ tất cả các ảnh
features = []
valid_paths = []  # Lưu trữ đường dẫn ảnh hợp lệ
for path in image_paths:
    feature = extract_features(path, model)
    if feature is not None:
        features.append(feature)
        valid_paths.append(path)

# Chuyển danh sách đặc trưng thành numpy array
features = np.array(features)

# Áp dụng DBSCAN để tìm ảnh trùng
dbscan = DBSCAN(eps=5, min_samples=2, metric='euclidean')  # eps có thể cần điều chỉnh
clusters = dbscan.fit_predict(features)

# Lưu kết quả vào DataFrame
cluster_df = pd.DataFrame({
    "ImagePath": valid_paths,
    "ClusterID": clusters
})

# Lọc các cluster có nhiều hơn 1 ảnh
duplicate_clusters = cluster_df[cluster_df["ClusterID"] != -1].groupby("ClusterID")
for cluster_id, group in duplicate_clusters:
    cluster_images = group["ImagePath"].tolist()
    print(f"\nCluster {cluster_id}:")
    print("\n".join(cluster_images))

    # Ghi kết quả vào danh sách all_results
    for img_path in cluster_images:
        all_results.append({
            "ClusterID": cluster_id,
            "ImagePath": img_path
        })


In [ ]:
# In ra tổng số lượng ảnh trùng
total_duplicates = len(all_results)
print(f"\nTổng số lượng ảnh trùng: {total_duplicates}")


Tổng số lượng ảnh trùng: 1334


In [ ]:
# Ghi toàn bộ kết quả vào file CSV
output_csv = '/content/drive/MyDrive/CS114.P11.CarData/0507clusters_train1'
result_df = pd.DataFrame(all_results)
result_df.to_csv(output_csv, index=False)
print(f"\nDuplicate detection results saved to {output_csv}")


Duplicate detection results saved to /content/drive/MyDrive/CS114.P11.CarData/0507clusters_train1


In [ ]:
def plot_clusters_in_rows(all_results):
    """
    Hiển thị các cụm ảnh trùng trên cùng một dòng cho mỗi cụm.
    """
    # Nhóm kết quả theo ClusterID
    clusters = {}
    for item in all_results:
        cluster_id = item['ClusterID']
        image_path = item['ImagePath']
        if cluster_id not in clusters:
            clusters[cluster_id] = []
        clusters[cluster_id].append(image_path)

    # Hiển thị từng cụm, mỗi cụm trên 1 hàng
    for cluster_id, image_paths in clusters.items():
        print(f"\nCluster ID: {cluster_id} (Total images: {len(image_paths)})")

        n_images = len(image_paths)

        # Tạo lưới ảnh với 1 hàng (n_rows=1)
        fig, axes = plt.subplots(1, n_images, figsize=(n_images * 2, 3), dpi=300)

        # Nếu chỉ có 1 ảnh, axes không phải danh sách
        if n_images == 1:
            axes = [axes]

        for i, image_path in enumerate(image_paths):
            try:
                img = load_img(image_path)  # Hàm load ảnh từ đường dẫn
                axes[i].imshow(img)
                axes[i].axis('off')  # Tắt trục tọa độ
                axes[i].set_title(os.path.basename(image_path), fontsize=8)
            except Exception as e:
                print(f"Error loading image {image_path}: {e}")

        plt.tight_layout()
        plt.show()


In [ ]:
plot_clusters_in_rows(all_results)